In [2]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import Callback, LearningRateScheduler
import time
import os


np.random.seed(42)
tf.random.set_seed(42)


class HessianDropout(layers.Layer):
    """
    Hessian Matrix Dropout Layer
    WSR = ||gradient|| / (||weight|| * ||Hessian||_F)
    Dropout probability: p_drop = sigmoid(alpha * WSR + beta)
    """

    def __init__(self, rate=0.5, alpha=1.0, beta=-1.0, eps=1e-8,
                 warmup_epochs=10, m_samples=1, **kwargs):
        super(HessianDropout, self).__init__(**kwargs)
        self.rate = rate
        self.alpha = alpha
        self.beta = beta
        self.eps = eps
        self.warmup_epochs = warmup_epochs
        self.m_samples = m_samples  # number of Hutchinson samples
        self.current_epoch = 0
        self._hes_norm = None
        self._wsr_values = None

    def build(self, input_shape):
        self.input_dim = input_shape[-1]
        super(HessianDropout, self).build(input_shape)

    def _compute_hessian_norm(self, inputs, gradients):
        """
        tr(H) ≈ v^T H v for random vector v
        For layer-wise estimation:
        ||H_l||_F ≈ sqrt((1/m) * sum((v_j^T H_l v_j)^2))
        """
        batch_size = tf.shape(inputs)[0]
        hessian_norm_sq = 0.0

        for _ in range(self.m_samples):
            v = tf.random.uniform(shape=tf.shape(inputs),minval=0, maxval=2, dtype=tf.float32)
            v = tf.cast(v > 1.0, tf.float32)
            v = 2.0 * v - 1.0

            with tf.GradientTape(persistent=False) as tape2:
                tape2.watch(inputs)
                with tf.GradientTape() as tape1:
                    tape1.watch(inputs)
                    loss = tf.reduce_sum(tf.square(inputs))
                grad = tape1.gradient(loss, inputs)
            hv = gradients * v
            vthv = tf.reduce_sum(v * hv)
            hessian_norm_sq += tf.square(vthv)
        hessian_norm = tf.sqrt(hessian_norm_sq / tf.cast(self.m_samples, tf.float32) + self.eps)
        return hessian_norm

    def call(self, inputs, training=None):
        if not training:
            return inputs
        if self.current_epoch < self.warmup_epochs:
            return K.dropout(inputs, self.rate)
        with tf.GradientTape(persistent=False) as tape:
            tape.watch(inputs)
            loss = tf.reduce_sum(tf.square(inputs))
        gradients = tape.gradient(loss, inputs)
        if gradients is None:
            return K.dropout(inputs, self.rate)

        hessian_norm = self._compute_hessian_norm(inputs, gradients)
        self._hes_norm = hessian_norm
        grad_norm = tf.abs(gradients) + self.eps
        input_norm = tf.abs(inputs) + self.eps

        # WSR per neuron (element-wise)
        wsr = grad_norm / (input_norm * (hessian_norm + self.eps))
        wsr_mean = tf.reduce_mean(wsr, axis=-1, keepdims=True)
        wsr_std = tf.math.reduce_std(wsr, axis=-1, keepdims=True) + self.eps
        wsr_norm = (wsr - wsr_mean) / wsr_std
        self._wsr_values = wsr_norm
        p_drop = tf.sigmoid(self.alpha * wsr_norm + self.beta)
        uniform = tf.random.uniform(shape=tf.shape(inputs))
        mask = tf.cast(uniform > p_drop, tf.float32)  # 1=keep, 0=drop
        keep_prob = 1.0 - p_drop
        scale = 1.0 / (keep_prob + self.eps)
        output = inputs * mask * scale
        return output

    def get_config(self):
        config = super(HessianDropout, self).get_config()
        config.update({
            'rate': self.rate,
            'alpha': self.alpha,
            'beta': self.beta,
            'eps': self.eps,
            'warmup_epochs': self.warmup_epochs,
            'm_samples': self.m_samples
        })
        return config


class HessianDropoutEpochCallback(Callback):

    def __init__(self, dropout_layers):
        super(HessianDropoutEpochCallback, self).__init__()
        self.dropout_layers = dropout_layers

    def on_epoch_begin(self, epoch, logs=None):
        for layer in self.dropout_layers:
            if hasattr(layer, 'current_epoch'):
                layer.current_epoch = epoch

    def on_epoch_end(self, epoch, logs=None):
        for layer in self.dropout_layers:
            if hasattr(layer, '_wsr_values') and layer._wsr_values is not None:
                wsr_mean = tf.reduce_mean(layer._wsr_values).numpy()
                wsr_std = tf.math.reduce_std(layer._wsr_values).numpy()

def build_model(input_shape=(28, 28, 1), num_classes=10):
    model = models.Sequential([
        layers.Flatten(input_shape=input_shape, name='flatten'),
        layers.Dense(200, activation='relu', name='dense1'),
        HessianDropout(rate=0.5, alpha=1.0, beta=-1.0, warmup_epochs=10,m_samples=1,name='hesdrop1'),
        layers.Dense(200, activation='relu', name='dense2'),
        HessianDropout(rate=0.5, alpha=1.0, beta=-1.0, warmup_epochs=10,m_samples=1,name='hesdrop2'),
        layers.Dense(num_classes, activation='softmax', name='output')
    ])

    return model


def lr_schedule(epoch):
    base_lr = 0.001
    reduction = 0.0005 * (epoch // 25)
    return max(base_lr - reduction, 0.0001)


def main():
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    x_train = np.expand_dims(x_train, axis=-1)
    x_test = np.expand_dims(x_test, axis=-1)
    y_train = keras.utils.to_categorical(y_train, 10)
    y_test = keras.utils.to_categorical(y_test, 10)
    print(f"  Train shape: {x_train.shape}")
    print(f"  Test shape: {x_test.shape}")
    model = build_model()
    hes_drop_layers = [layer for layer in model.layers
                       if isinstance(layer, HessianDropout)]
    model.summary()

    optimizer = Adam(learning_rate=0.001)
    loss_fn = CategoricalCrossentropy()
    model.compile(optimizer=optimizer,loss=loss_fn,metrics=['accuracy'])
    callbacks = [HessianDropoutEpochCallback(hes_drop_layers),LearningRateScheduler(lr_schedule)]

    start_time = time.time()

    history = model.fit(
        x_train, y_train,
        batch_size=64,
        epochs=50,
        validation_data=(x_test, y_test),
        callbacks=callbacks,
        verbose=1
    )

    train_time = time.time() - start_time
    print(f"\n  Training completed in {train_time:.2f} seconds")

    print("\n[5] Evaluating...")
    train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

    print(f"Train Accuracy:  {train_acc * 100:.2f}%")
    print(f"Test Accuracy:   {test_acc * 100:.2f}%")
    print(f"Gap:             {(train_acc - test_acc) * 100:.2f}%")
    total_params = model.count_params()
    flops_per_batch = (2 + 4 + 4) * total_params  # forward + backward + hessian
    gflops_per_batch = flops_per_batch / 1e9
    print(f"Estimated GFLOPs per batch: {gflops_per_batch:.6f}")

    return model, history


if __name__ == "__main__":
    model, history = main()



  Train shape: (60000, 28, 28, 1)
  Test shape: (10000, 28, 28, 1)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 200)            │       157,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hesdrop1 (HessianDropout)       │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (Dense)                  │ (None, 200)            │        40,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hesdrop2 (HessianDropout)       │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 199,210 (778.16 KB)

 Trainable params: 199,210 (778.16 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.8612 - loss: 0.4526 - val_accuracy: 0.9539 - val_loss: 0.1494 - learning_rate: 0.0010
Epoch 2/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9338 - loss: 0.2268 - val_accuracy: 0.9642 - val_loss: 0.1158 - learning_rate: 0.0010
Epoch 3/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9458 - loss: 0.1829 - val_accuracy: 0.9704 - val_loss: 0.0992 - learning_rate: 0.0010
Epoch 4/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9519 - loss: 0.1620 - val_accuracy: 0.9715 - val_loss: 0.0933 - learning_rate: 0.0010
Epoch 5/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9554 - loss: 0.1474 - val_accuracy: 0.9744 - val_loss: 0.0839 - learning_rate: 0.0010
Epoch 6/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9602 - loss: 0.1337 - val_accuracy: 0.9757 - val_loss: 0.0808 - learning_rate: 0.0010
Epoch 7/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9625 - loss: 0.1251 - 